In [1]:
import duckdb 
import polars as pl 

In [2]:
# خواندن فایل 
CSV_PATH = 'digikala-products.csv'
NULL_TOKENS = ['null', 'NULL', 'Null', 'NaN', 'nan', 'NAN', 'None', 'NA', 'N/A', '']
df2 = pl.read_csv(
    CSV_PATH,
    null_values=NULL_TOKENS,
)
df2.null_count()

id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,210108,0,0,223,0,0,0


In [3]:
df2 = df2.with_columns(
    pl.col('Category2').fill_null(pl.col('Category1'))
)

In [4]:
df2 = df2.with_columns(
    pl.col('Seller').fill_null('نامشخص')
)


In [5]:
# اضافه کردن ستون تفاوت قیمت امروز با حداقل قیمت ماه گذشته
df2 = df2.with_columns(
    (pl.col('Price') - pl.col('min_price_last_month')).alias('price_different'))

In [6]:
df2

id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category,price_different
i64,str,i64,i64,str,str,str,i64,str,bool,i64,str,i64
7096438,"""آبسلانگ مدل s5 بسته 250 عددی""",90,4,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",634800,"""سلامت ساز راد""",false,0,"""beauty""",634800
2845119,""" آبسلانگ مدل M-1 بسته 400 عددی""",84,217,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",818800,"""مهر افزون""",false,0,"""beauty""",818800
6117745,"""آبسلانگ مدل m50 مجموعه 500 عدد…",88,14,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",920000,"""یانگوم""",false,0,"""beauty""",920000
1912926,"""استند ابسلانگ مدل S01""",78,25,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",1100000,"""بهراد طب بیدار""",false,0,"""beauty""",1100000
6335462,"""آبسلانگ نوری تسلامد مدل All-in…",84,25,"""آبسلانگ""","""آبسلانگ""","""تسلا مد""",1530000,"""دیجی‌کالا""",false,0,"""beauty""",1530000
…,…,…,…,…,…,…,…,…,…,…,…,…
8745218,"""جعبه نگهدارنده دارو مدل چوبی""",90,24,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2500000,"""وین شاپ""",false,0,"""beauty""",2500000
8159085,"""محفظه نگهداری قرص و پودر میوتن…",96,4,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2150000,"""گیشا اسپورت""",false,0,"""beauty""",2150000
8137567,"""محفظه نگهداری قرص و پودر اسپرت…",70,2,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2100000,"""گیشا اسپورت""",false,0,"""beauty""",2100000


In [7]:
import polars as pl

n_total = df2.height
n_unique_rows = df2.unique().height
print('total rows      :', n_total)
print('unique rows     :', n_unique_rows)
print('duplicate rows  :', n_total - n_unique_rows)
print('duplicate %     :', round((n_total - n_unique_rows) / n_total * 100, 2))

# تکرار به تفکیک هر ستون
print(df2.select(
    [(pl.lit(n_total) - pl.col(c).n_unique()).alias(c) for c in df2.columns]
))


total rows      : 1283496
unique rows     : 960367
duplicate rows  : 323129
duplicate %     : 25.18
shape: (1, 13)
┌────────┬──────────┬─────────┬──────────┬───┬─────────┬──────────────┬──────────────┬─────────────┐
│ id     ┆ title_fa ┆ Rate    ┆ Rate_cnt ┆ … ┆ Is_Fake ┆ min_price_la ┆ sub_category ┆ price_diffe │
│ ---    ┆ ---      ┆ ---     ┆ ---      ┆   ┆ ---     ┆ st_month     ┆ ---          ┆ rent        │
│ u32    ┆ u32      ┆ u32     ┆ u32      ┆   ┆ u32     ┆ ---          ┆ u32          ┆ ---         │
│        ┆          ┆         ┆          ┆   ┆         ┆ u32          ┆              ┆ u32         │
╞════════╪══════════╪═════════╪══════════╪═══╪═════════╪══════════════╪══════════════╪═════════════╡
│ 335144 ┆ 376182   ┆ 1283445 ┆ 1280373  ┆ … ┆ 1283494 ┆ 1266639      ┆ 1283490      ┆ 1237780     │
└────────┴──────────┴─────────┴──────────┴───┴─────────┴──────────────┴──────────────┴─────────────┘


In [10]:
import polars as pl

n_total = df2.height
df_uniq = df2.unique()

# ۱) ردیف‌های کاملاً تکراری
n_exact_dup = n_total - df_uniq.height

# ۲) id هایی که بیش از یک بار آمده‌اند
n_id_unique = df2['id'].n_unique()
n_id_dup = df2.group_by('id').len().filter(pl.col('len') > 1).height

# ۳) id های تکراری با اطلاعات متفاوت (بعد از حذف کپی کامل)
n_id_diff = df_uniq.group_by('id').len().filter(pl.col('len') > 1).height

print(f"کل ردیف‌ها                    : {n_total:,}")
print(f"id یکتا                       : {n_id_unique:,}")
print(f"1) ردیف کاملاً تکراری         : {n_exact_dup:,}  ({n_exact_dup/n_total*100:.2f}% از کل ردیف‌ها)")
print(f"2) id تکراری                  : {n_id_dup:,}  ({n_id_dup/n_id_unique*100:.2f}% از id های یکتا)")
print(f"3) id تکراری با اطلاعات متفاوت: {n_id_diff:,}  ({n_id_diff/n_id_unique*100:.2f}% از id های یکتا)")


کل ردیف‌ها                    : 1,283,496
id یکتا                       : 948,352
1) ردیف کاملاً تکراری         : 323,129  (25.18% از کل ردیف‌ها)
2) id تکراری                  : 258,718  (27.28% از id های یکتا)
3) id تکراری با اطلاعات متفاوت: 11,785  (1.24% از id های یکتا)


In [12]:
import polars as pl

before_rows = df2.height
before_mb = df2.estimated_size('mb')
before_ids = df2['id'].n_unique()

df2 = df2.unique(maintain_order=True)

after_rows = df2.height
after_mb = df2.estimated_size('mb')
after_ids = df2['id'].n_unique()

print(f"ردیف قبل   : {before_rows:,}")
print(f"ردیف بعد   : {after_rows:,}")
print(f"حذف‌شده    : {before_rows - after_rows:,} ({(before_rows - after_rows) / before_rows * 100:.2f}%)")
print(f"حجم قبل    : {before_mb:,.1f} MB")
print(f"حجم بعد    : {after_mb:,.1f} MB")
print(f"id یکتا    : {before_ids:,} / {after_ids:,}")


ردیف قبل   : 1,283,496
ردیف بعد   : 960,367
حذف‌شده    : 323,129 (25.18%)
حجم قبل    : 268.5 MB
حجم بعد    : 201.5 MB
id یکتا    : 948,352 / 948,352


In [15]:
df2 = df2.with_columns(
    pl.when(pl.col('Rate').is_null()).then(pl.lit(None))
      .when(pl.col('Rate') > 80).then(pl.lit('1_کالای با کیفیت'))
      .when(pl.col('Rate') >= 60).then(pl.lit('2_کالای خوب'))
      .when(pl.col('Rate') >= 40).then(pl.lit('3_کالای معمولی'))
      .otherwise(pl.lit('4_کالای بی کیفیت'))
      .alias('quality_level')
)

print(df2['quality_level'].value_counts().sort('quality_level'))


shape: (4, 2)
┌─────────────────┬────────┐
│ quality_level   ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ 1_کالای با کیفیت ┆ 183984 │
│ 2_کالای خوب      ┆ 160196 │
│ 3_کالای معمولی   ┆ 18056  │
│ 4_کالای بی کیفیت ┆ 598131 │
└─────────────────┴────────┘


In [16]:
df2

id,title_fa,Rate,Rate_cnt,Category1,Category2,Brand,Price,Seller,Is_Fake,min_price_last_month,sub_category,price_different,quality_level
i64,str,i64,i64,str,str,str,i64,str,bool,i64,str,i64,str
7096438,"""آبسلانگ مدل s5 بسته 250 عددی""",90,4,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",634800,"""سلامت ساز راد""",false,0,"""beauty""",634800,"""1_کالای با کیفیت"""
2845119,""" آبسلانگ مدل M-1 بسته 400 عددی""",84,217,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",818800,"""مهر افزون""",false,0,"""beauty""",818800,"""1_کالای با کیفیت"""
6117745,"""آبسلانگ مدل m50 مجموعه 500 عدد…",88,14,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",920000,"""یانگوم""",false,0,"""beauty""",920000,"""1_کالای با کیفیت"""
1912926,"""استند ابسلانگ مدل S01""",78,25,"""آبسلانگ""","""آبسلانگ""","""متفرقه""",1100000,"""بهراد طب بیدار""",false,0,"""beauty""",1100000,"""2_کالای خوب"""
6335462,"""آبسلانگ نوری تسلامد مدل All-in…",84,25,"""آبسلانگ""","""آبسلانگ""","""تسلا مد""",1530000,"""دیجی‌کالا""",false,0,"""beauty""",1530000,"""1_کالای با کیفیت"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
8745218,"""جعبه نگهدارنده دارو مدل چوبی""",90,24,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2500000,"""وین شاپ""",false,0,"""beauty""",2500000,"""1_کالای با کیفیت"""
8159085,"""محفظه نگهداری قرص و پودر میوتن…",96,4,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2150000,"""گیشا اسپورت""",false,0,"""beauty""",2150000,"""1_کالای با کیفیت"""
8137567,"""محفظه نگهداری قرص و پودر اسپرت…",70,2,"""یادآور و نگهدارنده قرص و دارو""","""یادآور و نگهدارنده قرص و دارو""","""متفرقه""",2100000,"""گیشا اسپورت""",false,0,"""beauty""",2100000,"""2_کالای خوب"""
